# Canadian Household Spending Analysis — 2021 vs 2023 Official Bridge

**Analyst:** Carlos Restrepo | GLOCAL Foundation of Canada  
**Phase:** Comparative Extension — 2021 PUMF vs 2023 Official Table  
**Source:**  
- 2021 validated project outputs from SHS PUMF analysis  
- 2023 official Statistics Canada table 11-10-0222-01  
**Kernel:** Python (shs2021)

## Purpose
This notebook compares the validated 2021 national SHS PUMF results against the 2023 official aggregate table in order to identify what changed after the COVID-affected 2021 year.

## Important methodological note
This notebook compares:
- **2021 PUMF-based national results**, already validated in the project
- **2023 official aggregate results**, taken from Statistics Canada published tables

This is a valid analytical bridge, but it is **not** a perfect like-for-like microdata comparison.  
The comparison is strongest for:
- national expenditure levels
- food, shelter, and transport
- food channel split (stores vs restaurants)
- provincial ranking patterns for shelter, food, and transport

This notebook does **not** reproduce 2023 quintile or tenure analysis because those dimensions are not available in table 11-10-0222-01.


In [ ]:
import os
# ============================================================
# Cell 2 - Import libraries and load 2023 official table
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

data_path = r"C:\Users\LENOVO\canadian_household_spending\outputs\2023\11100222.csv"
df23_raw = pd.read_csv(data_path)

print("OK 2023 official table loaded!")
print("Rows   :", f"{len(df23_raw):,}")
print("Cols   :", len(df23_raw.columns))
print("\nYears available:")
print(sorted(df23_raw["REF_DATE"].dropna().unique().tolist()))


In [ ]:
# ============================================================
# Cell 3 - Prepare 2023 official Canada + province table
# ============================================================

cat_col = "Household expenditures, summary-level categories"

keep_geos = [
    "Canada",
    "Newfoundland and Labrador",
    "Prince Edward Island",
    "Nova Scotia",
    "New Brunswick",
    "Quebec",
    "Ontario",
    "Manitoba",
    "Saskatchewan",
    "Alberta",
    "British Columbia",
]

key_cats = [
    "Total expenditure",
    "Total current consumption",
    "Food expenditures",
    "Food purchased from stores",
    "Food purchased from restaurants",
    "Shelter",
    "Transportation",
]

df23 = df23_raw[
    (df23_raw["REF_DATE"] == 2023) &
    (df23_raw["Statistic"] == "Average expenditure per household") &
    (df23_raw["GEO"].isin(keep_geos)) &
    (df23_raw[cat_col].isin(key_cats))
].copy()

print("Filtered shape:", df23.shape)
print("\nGeographies kept:")
print(sorted(df23["GEO"].unique()))


In [ ]:
# ============================================================
# Cell 4 - Validated 2021 national anchor + 2023 national summary
# ============================================================

national_2021 = pd.DataFrame({
    "Year": [2021],
    "Avg_TotalExp": [90185.1],
    "Avg_Food": [10305.6],
    "Food_Stores": [8063.5],
    "Food_Restaurants": [2190.3],
    "Avg_Shelter": [20456.9],
    "Avg_Transport": [10289.1],
    "Stores_Share_%": [78.2],
    "Restaurants_Share_%": [21.3],
})

nat23 = (
    df23[df23["GEO"] == "Canada"]
    .pivot_table(index="GEO", columns=cat_col, values="VALUE", aggfunc="first")
    .reset_index(drop=True)
)

national_2023 = pd.DataFrame({
    "Year": [2023],
    "Avg_TotalExp": [nat23["Total expenditure"].iloc[0]],
    "Avg_CurrentConsumption": [nat23["Total current consumption"].iloc[0]],
    "Avg_Food": [nat23["Food expenditures"].iloc[0]],
    "Food_Stores": [nat23["Food purchased from stores"].iloc[0]],
    "Food_Restaurants": [nat23["Food purchased from restaurants"].iloc[0]],
    "Avg_Shelter": [nat23["Shelter"].iloc[0]],
    "Avg_Transport": [nat23["Transportation"].iloc[0]],
})

national_2023["Stores_Share_%"] = national_2023["Food_Stores"] / national_2023["Avg_Food"] * 100
national_2023["Restaurants_Share_%"] = national_2023["Food_Restaurants"] / national_2023["Avg_Food"] * 100

print("=== 2021 NATIONAL ANCHOR ===")
display(national_2021.round(1))

print("=== 2023 NATIONAL SUMMARY ===")
display(national_2023.round(1))


In [ ]:
# ============================================================
# Cell 5 - Bridge table: 2021 PUMF vs 2023 official
# ============================================================

bridge_2021_2023 = pd.DataFrame({
    "Metric": [
        "Avg total expenditure",
        "Avg food expenditure",
        "Food from stores",
        "Food from restaurants",
        "Avg shelter expenditure",
        "Avg transport expenditure",
        "Stores share %",
        "Restaurants share %"
    ],
    "2021_PUMF": [
        90185.1,
        10305.6,
        8063.5,
        2190.3,
        20456.9,
        10289.1,
        78.2,
        21.3
    ],
    "2023_Official": [
        national_2023["Avg_TotalExp"].iloc[0],
        national_2023["Avg_Food"].iloc[0],
        national_2023["Food_Stores"].iloc[0],
        national_2023["Food_Restaurants"].iloc[0],
        national_2023["Avg_Shelter"].iloc[0],
        national_2023["Avg_Transport"].iloc[0],
        national_2023["Stores_Share_%"].iloc[0],
        national_2023["Restaurants_Share_%"].iloc[0]
    ]
})

bridge_2021_2023["Absolute_Change"] = bridge_2021_2023["2023_Official"] - bridge_2021_2023["2021_PUMF"]
bridge_2021_2023["Pct_Change"] = (bridge_2021_2023["Absolute_Change"] / bridge_2021_2023["2021_PUMF"]) * 100

print("=== 2021 vs 2023 NATIONAL BRIDGE ===")
print(bridge_2021_2023.round(1).to_string(index=False))


In [ ]:
# ============================================================
# Cell 6 - 2021 province anchor vs 2023 province official table
# ============================================================

prov_2021 = pd.DataFrame([
    ["Alberta", 121256.0, 11901.0, 22056.0, 10936.0, 18.2, 9.8, 9.0],
    ["British Columbia", 103654.0, 11341.0, 24139.0, 11033.0, 23.3, 10.9, 10.6],
    ["Manitoba", 91981.0, 9855.0, 17332.0, 11848.0, 18.8, 10.7, 12.9],
    ["New Brunswick", 82883.0, 9942.0, 13970.0, 10070.0, 16.9, 12.0, 12.1],
    ["Newfoundland and Labrador", 92024.0, 11505.0, 15866.0, 11203.0, 17.2, 12.5, 12.2],
    ["Nova Scotia", 88338.0, 9386.0, 16802.0, 10987.0, 19.0, 10.6, 12.4],
    ["Ontario", 110189.0, 9822.0, 23276.0, 9734.0, 21.1, 8.9, 8.8],
    ["Prince Edward Island", 84267.0, 10395.0, 16514.0, 10516.0, 19.6, 12.3, 12.5],
    ["Quebec", 90521.0, 9731.0, 15313.0, 9965.0, 16.9, 10.7, 11.0],
    ["Saskatchewan", 97541.0, 11736.0, 18884.0, 11525.0, 19.4, 12.0, 11.8],
], columns=[
    "Province", "Avg_Income", "Avg_Food", "Avg_Shelter", "Avg_Transport",
    "Shelter_pct_income", "Food_pct_income", "Transport_pct_income"
])

prov23_pivot = (
    df23[df23["GEO"] != "Canada"]
    .pivot_table(index="GEO", columns=cat_col, values="VALUE", aggfunc="first")
    .reset_index()
)

prov_2023 = prov23_pivot.rename(columns={
    "GEO": "Province",
    "Total expenditure": "Avg_TotalExp",
    "Total current consumption": "Avg_CurrentConsumption",
    "Food expenditures": "Avg_Food",
    "Food purchased from stores": "Food_Stores",
    "Food purchased from restaurants": "Food_Restaurants",
    "Shelter": "Avg_Shelter",
    "Transportation": "Avg_Transport"
}).copy()

prov_2023["Shelter_pct_consumption"] = ((prov_2023["Avg_Shelter"] / prov_2023["Avg_CurrentConsumption"]) * 100).round(1)
prov_2023["Food_pct_consumption"] = ((prov_2023["Avg_Food"] / prov_2023["Avg_CurrentConsumption"]) * 100).round(1)
prov_2023["Transport_pct_consumption"] = ((prov_2023["Avg_Transport"] / prov_2023["Avg_CurrentConsumption"]) * 100).round(1)

print("=== 2021 PROVINCIAL ANCHOR ===")
display(prov_2021)

print("=== 2023 PROVINCIAL OFFICIAL TABLE ===")
display(prov_2023[[
    "Province", "Avg_CurrentConsumption", "Avg_Food", "Avg_Shelter", "Avg_Transport",
    "Shelter_pct_consumption", "Food_pct_consumption", "Transport_pct_consumption"
]].round(1))


## Provincial comparison rule

The provincial comparison is valid for **ranking and pattern**, but not for exact burden point changes.

- **2021** uses shelter, food, and transport as a **share of income**
- **2023** uses shelter, food, and transport as a **share of current consumption**

This means the notebook can compare:
- which provinces remain near the top
- whether the same high-pressure housing markets persist
- whether food and transport still take relatively large shares in some provinces

This notebook should **not** interpret the provincial percentages as a direct point-to-point burden change.


In [ ]:
# ============================================================
# Cell 7 - Charts: national bridge
# ============================================================

plot_df = bridge_2021_2023.copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("2021 PUMF vs 2023 Official — National Bridge", fontsize=14, fontweight="bold")

main_metrics = plot_df[plot_df["Metric"].isin([
    "Avg total expenditure",
    "Avg food expenditure",
    "Avg shelter expenditure",
    "Avg transport expenditure"
])]

axes[0,0].bar(main_metrics["Metric"], main_metrics["2021_PUMF"])
axes[0,0].set_title("2021 PUMF", fontweight="bold")
axes[0,0].tick_params(axis='x', rotation=45)

axes[0,1].bar(main_metrics["Metric"], main_metrics["2023_Official"])
axes[0,1].set_title("2023 Official", fontweight="bold")
axes[0,1].tick_params(axis='x', rotation=45)

channel_metrics = plot_df[plot_df["Metric"].isin([
    "Food from stores",
    "Food from restaurants"
])]

axes[1,0].bar(channel_metrics["Metric"], channel_metrics["2021_PUMF"])
axes[1,0].set_title("Food Channels — 2021", fontweight="bold")
axes[1,0].tick_params(axis='x', rotation=45)

axes[1,1].bar(channel_metrics["Metric"], channel_metrics["2023_Official"])
axes[1,1].set_title("Food Channels — 2023", fontweight="bold")
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 8 - Charts: provincial shelter pattern
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
fig.suptitle("Provincial Shelter Pattern — 2021 vs 2023", fontsize=14, fontweight="bold")

plot21 = prov_2021.sort_values("Shelter_pct_income")
sns.barplot(data=plot21, y="Province", x="Shelter_pct_income", palette="Blues_d", ax=axes[0])
axes[0].set_title("2021 Shelter Burden as % of Income", fontweight="bold")
axes[0].set_xlabel("% of income")
axes[0].set_ylabel("")

plot23 = prov_2023.sort_values("Shelter_pct_consumption")
sns.barplot(data=plot23, y="Province", x="Shelter_pct_consumption", palette="Oranges_d", ax=axes[1])
axes[1].set_title("2023 Shelter Share of Current Consumption", fontweight="bold")
axes[1].set_xlabel("% of current consumption")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 9 - Food subcategory extension for 2023
# ============================================================

food_subcats = [
    "Bakery products",
    "Cereal grains and cereal products",
    "Fruit, fruit preparations and nuts",
    "Vegetables and vegetable preparations",
    "Dairy products and eggs",
    "Meat",
    "Fish and seafood",
    "Non-alcoholic beverages and other food products",
]

food23 = (
    df23[(df23["GEO"] == "Canada") & (df23[cat_col].isin(food_subcats))]
    [[cat_col, "VALUE"]]
    .rename(columns={cat_col: "Subcategory", "VALUE": "Avg_Spend"})
    .sort_values("Avg_Spend", ascending=False)
)

food23["Share_of_Total_Food_%"] = (food23["Avg_Spend"] / national_2023["Avg_Food"].iloc[0]) * 100

print("=== 2023 FOOD SUBCATEGORY EXTENSION ===")
print(food23.round(1).to_string(index=False))


In [ ]:
# ============================================================
# Cell 10 - Key findings
# ============================================================

print("=== 2021 vs 2023 BRIDGE FINDINGS ===\n")
print("1. Total household expenditure increased strongly from 2021 to 2023.")
print("2. Shelter continued to rise and remained the largest structural spending category.")
print("3. Food increased overall, but the strongest growth came from restaurants rather than stores.")
print("4. The store-heavy pattern observed in 2021 softened in 2023 as restaurant spending rebounded.")
print("5. Transport also rebounded, supporting the interpretation that 2021 transport spending was still compressed by COVID-era conditions.")
print("6. At the provincial level, Ontario and British Columbia remained among the most shelter-intensive markets.")
print("7. The provincial comparison should be read as a ranking and pattern comparison, not as a direct burden-point change, because the 2021 and 2023 denominators are different.")


## Interpretation notes

This bridge notebook supports three important conclusions.

First, several of the strongest 2021 patterns were at least partly shaped by COVID-era conditions rather than representing a stable long-run endpoint. This is clearest in food channels and transportation.

Second, shelter remained the most structurally important category after 2021. Even though the 2023 provincial measure uses current consumption rather than income, the same housing-intensive markets remain near the top of the provincial ranking.

Third, the 2023 extension is analytically useful even without quintile and tenure tables because it shows what happened to the national structure of household spending after the 2021 reference year.


In [ ]:
# ============================================================
# Cell 11 - Save outputs
# ============================================================

from pathlib import Path

out_dir = Path(r"C:\Users\LENOVO\canadian_household_spending\outputs\2023")
out_dir.mkdir(parents=True, exist_ok=True)

bridge_2021_2023.to_csv(out_dir / "bridge_2021_2023_national.csv", index=False)
prov_2021.to_csv(out_dir / "bridge_2021_provincial_anchor.csv", index=False)
prov_2023.to_csv(out_dir / "bridge_2023_provincial_official.csv", index=False)
food23.to_csv(out_dir / "bridge_2023_food_subcategories.csv", index=False)

print("OK Bridge outputs saved!")


In [ ]:
import pandas as pd
import urllib
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

connection_url = URL.create(
    "mssql+pyodbc",
    username = os.environ.get("AZURE_SQL_USERNAME", "your_username")
    password = os.environ.get("AZURE_SQL_PASSWORD", "YOUR_PASSWORD_HERE"),
    server   = os.environ.get("AZURE_SQL_SERVER", "your-server.database.windows.net")
    database = os.environ.get("AZURE_SQL_DATABASE", "your_database")
    query={
        "driver": "ODBC Driver 18 for SQL Server",
        "Encrypt": "yes",
        "TrustServerCertificate": "no",
        "Connection Timeout": "30",
    },
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    print(conn.execute(text("SELECT 1")).fetchall())

# Subir comparative bridge 2021 vs 2023
df23_raw.to_sql("comparative_bridge_raw", engine, if_exists="replace", index=False)
print(f"✓ comparative_bridge_raw subida: {df23_raw.shape}")

df23.to_sql("comparative_bridge_filtered", engine, if_exists="replace", index=False)
print(f"✓ comparative_bridge_filtered subida: {df23.shape}")